In [16]:
# 04_evaluation.ipynb

import os
import glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc
)
import seaborn as sns

# Asegúrate de importar correctamente tu clase de modelo
from src.models.cnn_model import ResNet1D

# 1. Cargar modelo (estructura + pesos)
model_path = '../models/cnn_model.pt'
from src.models.cnn_model import ResNet1D  # asegúrate que la ruta es correcta

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet1D().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# 2. Cargar curvas de luz desde data/processed
import numpy as np

def load_curves_from_folder(folder, label, expected_length=2002):
    files = glob.glob(os.path.join(folder, '*.csv'))
    data = []
    labels = []
    for file in files:
        df = pd.read_csv(file)
        flux = df['flux'].values
        
        if len(flux) >= expected_length:
            flux = flux[:expected_length]  # recortar si es más largo
        else:
            # rellenar con ceros si es más corto
            padding = np.zeros(expected_length - len(flux))
            flux = np.concatenate((flux, padding))
        
        data.append(flux)
        labels.append(label)
    return data, labels

pos_data, pos_labels = load_curves_from_folder('../data/processed/positive', 1)
neg_data, neg_labels = load_curves_from_folder('../data/processed/negative', 0)

X = np.array(pos_data + neg_data)
y = np.array(pos_labels + neg_labels)
print(f"Datos cargados: {len(X)} curvas, {len(y)} etiquetas")

# 3. Convertir a tensores (CNN espera [N, 1, 2001])
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y)

# 4. Evaluación
with torch.no_grad():
    outputs = model(X_tensor)
    probs = torch.sigmoid(outputs).squeeze().numpy()
    y_pred = (probs >= 0.5).astype(int)

# 5. Métricas seguras
print("Etiquetas reales:", np.unique(y))
print("Etiquetas predichas:", np.unique(y_pred))

from sklearn.utils.multiclass import unique_labels

labels = unique_labels(y, y_pred)
target_names = [f"Clase {label}" for label in labels]

try:
    print(classification_report(y, y_pred, labels=labels, target_names=target_names, digits=4))
except ValueError as e:
    print("⚠️ Error al generar classification_report:", e)
    print("Verifica que haya al menos una instancia de cada clase en y e y_pred.")

# Confusion matrix
conf_mat = confusion_matrix(y, y_pred)

# 6. Graficar matriz de confusión
plt.figure(figsize=(5, 4))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
fig_path = '../reports/figures/confusion_matrix/confusion_matrix.png'
txt_path = '../reports/figures/confusion_matrix/description.txt'
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path)

with open(txt_path, 'w') as f:
    f.write("Matriz de confusión para el modelo CNN evaluado.\n"
            "La fila representa la clase real y la columna la predicha.\n"
            "Valores: [TP, FP; FN, TN]")

# 7. ROC curve
fpr, tpr, _ = roc_curve(y, probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
roc_fig_path = '../reports/figures/roc_curve/roc_curve.png'
roc_txt_path = '../reports/figures/roc_curve/description.txt'
os.makedirs(os.path.dirname(roc_fig_path), exist_ok=True)
plt.savefig(roc_fig_path)

with open(roc_txt_path, 'w') as f:
    f.write("Curva ROC para el modelo CNN. Se muestra la relación entre TPR y FPR para diferentes umbrales.\n"
            f"AUC: {roc_auc:.4f}")


Datos cargados: 32 curvas, 32 etiquetas


RuntimeError: Expected 2D (unbatched) or 3D (batched) input to conv1d, but got input of size: [32, 1, 1, 2002]